In [39]:
# Célula 0 — Instalar bibliotecas necessárias
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# Celula 1 — Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


In [41]:
# Célula 2 — Carregamento do dataset

# Opção 1: CSV
df = pd.read_csv("br_seeg_emissoes_brasil.csv")

# # Visualizar primeiras linhas
# df.head()

# Visualizar informações do dataset
print(f"Shape original: {df.shape}")
# df.shape serve para mostrar o número de linhas e colunas do DataFrame. O resultado é uma tupla (n_linhas, n_colunas).
print(df.info())


Shape original: (454850, 12)
<class 'pandas.DataFrame'>
RangeIndex: 454850 entries, 0 to 454849
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ano                  454850 non-null  int64  
 1   nivel_1              454850 non-null  str    
 2   nivel_2              454850 non-null  str    
 3   nivel_3              454850 non-null  str    
 4   nivel_4              454850 non-null  str    
 5   nivel_5              454850 non-null  str    
 6   nivel_6              454850 non-null  str    
 7   tipo_emissao         454850 non-null  str    
 8   gas                  454850 non-null  str    
 9   atividade_economica  453100 non-null  str    
 10  produto              268100 non-null  str    
 11  emissao              406325 non-null  float64
dtypes: float64(1), int64(1), str(10)
memory usage: 41.6 MB
None


In [42]:
# Célula 3 — Seleção das colunas relevantes

df = df[["ano", "nivel_1", "gas", "emissao"]]

df.head()

print(f"Shape após seleção: {df.shape}")

Shape após seleção: (454850, 4)


In [43]:
# Célula 4 — Filtrar apenas emissões de CO2

df_co2 = df[df["gas"] == "CO2 (t)"].copy() #.copy() é usado para criar uma cópia do DataFrame filtrado, evitando o aviso de "SettingWithCopyWarning" do pandas.
# modifições seguras sem afetar o dataframe original.

print(f"Shape após filtro: {df_co2.shape}")
print(f"\nValores nulos por coluna:")
print(df_co2.isnull().sum())


Shape após filtro: (37550, 4)

Valores nulos por coluna:
ano           0
nivel_1       0
gas           0
emissao    6755
dtype: int64


In [44]:
# Célula 5 — Remoção de valores nulos

# Verificando a quantidade de NaN na coluna "emissao"
total = len(df_co2)
nulos = df_co2["emissao"].isnull().sum()
print(f"Total de registros: {total}")
print(f"Valores nulos na coluna 'emissao': {nulos} ({100*nulos/total:.1f}%)")

'''
    Removendo linhas com valores nulos na coluna "emissao",
    porque ela é nossa variavél alvo (y). Modelos de regressão não conseguem aprender com valores NaN no target.
    
    NaN faz sentido para análise exploratória, mas para treinar modelos de machine learning é obrigatório remove-los.
'''

df_co2 = df_co2.dropna(subset=["emissao"])
print(f"\nApós remoção de NaN: ")
print(f"registros restantes: {len(df_co2)}")
print(f"Valores nulos restantes: {df_co2['emissao'].isnull().sum().sum()}")

Total de registros: 37550
Valores nulos na coluna 'emissao': 6755 (18.0%)

Após remoção de NaN: 
registros restantes: 30795
Valores nulos restantes: 0


In [45]:
# Célula 6 — Aplicação de One-Hot Encoding

'''
    Aplicamos One_Hot Encoding na coluna "nivel_1".
    get_dummies() é uma função do pandas que converte uma coluna categórica em várias colunas binárias (0 ou 1) para cada categoria única.
    drop_first=True é usado para evitar a multicolinearidade, removendo a primeira categoria como referência.
    (se todas dummies são 0, sabemos que é a cotegoria removida).
'''
if "gas" in df_co2.columns:
    df_co2 = df_co2.drop(columns=["gas"]) # removendo a coluna "gas" porque agora só temos CO2, então ela não tem mais informação útil para o modelo.
    print("Coluna 'gas' removida, pois só temos CO2.")
else:
    print("Coluna 'gas' já foi removida ou não existe.")
df_encoded = pd.get_dummies(
    df_co2,
    columns=["nivel_1"],
    drop_first=True, # drop_first=True é usado para evitar a multicolinearidade, removendo a primeira categoria como referência.
    dtype=int
)

print((f"Shape após One-Hot Encoding: {df_encoded.shape}"))
print(f"Colunas: {list(df_encoded.columns)}")
df_encoded.head()

''' 
Nivel_1 possui 5 categorias únicas, então o One-Hot Encoding criou 4 novas colunas (uma a menos por causa do drop_first=True):
    nivel_1_Energia	
    nivel_1_Mudança de Uso da Terra e Floresta	
    nivel_1_Processos Industriais	
    nivel_1_Resíduos
'''


Coluna 'gas' removida, pois só temos CO2.
Shape após One-Hot Encoding: (30795, 6)
Colunas: ['ano', 'emissao', 'nivel_1_Energia', 'nivel_1_Mudança de Uso da Terra e Floresta', 'nivel_1_Processos Industriais', 'nivel_1_Resíduos ']


' \nNivel_1 possui 5 categorias únicas, então o One-Hot Encoding criou 4 novas colunas (uma a menos por causa do drop_first=True):\n    nivel_1_Energia\t\n    nivel_1_Mudança de Uso da Terra e Floresta\t\n    nivel_1_Processos Industriais\t\n    nivel_1_Resíduos\n'

In [46]:
# Célula 7 — Separar X e y antes da normalização


'''
    Primeiro separamos, depois normalizamos
    Isso garante que X e y sempre tenham a mesma quantidade de linhas, evitando erros de alinhamento.
'''

# Features = tudo menos 'emissao'

X = df_encoded.drop(columns=["emissao"])
y = df_encoded["emissao"].copy()

# Conferência imediata
print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")

assert X.shape[0] == y.shape[0], "Número de linhas de X e y deve ser igual!"
print(f"\nTamanho iguais: {X.shape[0]} amostras ")
print(f"Colunas de X: {list(X.columns)}")

Shape de X: (30795, 5)
Shape de y: (30795,)

Tamanho iguais: 30795 amostras 
Colunas de X: ['ano', 'nivel_1_Energia', 'nivel_1_Mudança de Uso da Terra e Floresta', 'nivel_1_Processos Industriais', 'nivel_1_Resíduos ']


In [47]:
# Célula 8 - normalização Min-Max

'''
    Normalização Min-Max: transforma os valores para o intervalo [0, 1].
    Fórmula: x_norm = (x - x_min) / (x_max - x_min)
'''

# Normalizar as features (X)
scaler_X = MinMaxScaler()
X = pd.DataFrame(
    scaler_X.fit_transform(X),
    columns=X.columns,
    index=X.index # mantém o mesmo índice do DataFrame original.
)
# Normalizar a variável de resposta - target (y)
scaler_y = MinMaxScaler()
y = pd.DataFrame(
    
)

# Modelos de Regressão.

Regressão Linear Multipla.  Multiple Linear Regression (MLR)

In [ ]:
# Célula 9 — Divisão dos dados

X

ValueError: Found input variables with inconsistent numbers of samples: [30795, 0]

In [ ]:
# Célula 10 — Função auxiliar para avaliar modelos

def avaliar_modelo(nome, modelo, X_train, y_train, X_test, y_test):
    '''
    Treina o modelo, faz predições e retorna as métricas.
    
    Métricas:
        - MAE (Mean Absolute Error): Erro médio absoluto entre as predições e os valores reais. 
            Quanto menor, melhor.
        - RMSE (Root Mean Squared Error): Raiz do erro quadrático médio entre as predições e os valores reais. 
            Quanto menor, melhor.
        - R² (Coeficiente de Determinação): Proporção da variância da variável alvo que é explicada pelo modelo. 
            Quanto maior, melhor.
                -> R² = 1.0 : modelo perfeito. 0.0 : modelo que não explica nada. Negativo: modelo pior que a média.
    '''
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"{'='*50}") # 50 caracteres de "=" para separar os resultados
    print(f"{nome}")
    print(f"MAE: {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"R²: {r2:.6f}")
    
    return {"nome" : nome, "MAE": mae, "RMSE": rmse, "R2": r2, "y_pred": y_pred}

In [ ]:
# Célula 10.5 - Debug: Verificar se X está limpo

# Verifica se há colunas com texto (object/string)
print("Tipos de cada coluna em X: ")
print(X.dtypes)
print()

# Se aparecer "Object" ou "String" aqui, algo deu errado nas células anteriores
colunas_texto = X.select_dtypes(include=["object","string"]).columns.tolist()

if colunas_texto:
    print(f"    Problema! Colunas com texto encontradas: {colunas_texto}")
    print(f"    Essas colunas precisam ser removidas ou codificadas.")
    print(f"    Faça: Kernel -> Restart & Run All")
else:
    print(f"    Tudo certo! todas as colunas são numéricas.")

print(f"\nColunas de X: {list(X.columns)}")
X.head()

Tipos de cada coluna em X: 
ano                                           float64
nivel_1_Energia                                 int64
nivel_1_Mudança de Uso da Terra e Floresta      int64
nivel_1_Processos Industriais                   int64
nivel_1_Resíduos                                int64
dtype: object

    Tudo certo! todas as colunas são numéricas.

Colunas de X: ['ano', 'nivel_1_Energia', 'nivel_1_Mudança de Uso da Terra e Floresta', 'nivel_1_Processos Industriais', 'nivel_1_Resíduos ']


,ano,nivel_1_Energia,nivel_1_Mudança de Uso da Terra e Floresta,nivel_1_Processos Industriais,nivel_1_Resíduos
14150,0.000000,0,0,0,0
14151,0.020408,0,0,0,0
14152,0.040816,0,0,0,0
14153,0.061224,0,0,0,0
14154,0.081633,0,0,0,0


In [ ]:
# Célula 11 — Modelo 1: Regressão Linear Múltipla (MLR)

mlr = LinearRegression()
res_mlr = avaliar_modelo("Regressão Linear Múltipla", mlr, X_train, X_test, y_train, y_test)

ValueError: Found input variables with inconsistent numbers of samples: [24636, 6159]

In [ ]:
# Célula 12 - 